# Stage 1: Single Agent (LLM + Memory + Tools)




## Install dependencies
Run this once in a fresh environment.


In [ ]:
# !pip -q install langgraph langchain-openai python-dotenv

## 1) Imports

In [ ]:
import os
from dotenv import load_dotenv
from typing import Any, Dict, Optional
from typing import TypedDict, Annotated, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

## 2) Load environment variables - please read instructions carefully

In [ ]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
load_dotenv()

In [ ]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

## 3) System prompt

In [ ]:
SYSTEM = """You are a travel planning agent.

Rules:
- Ask clarifying questions only when essential info is missing.
- Do not invent facts. Use search tool for fresh facts when needed.
- When user asks "total cost", ALWAYS call estimate_trip_cost with the latest trip parameters from conversation.
- If user says "add additional one day trip", interpret as +1 day unless they explicitly say it replaces an existing day.

Output format:
1) Day-by-day plan (brief)
2) Total cost (with assumptions)
"""


## 4) Tool - estimate trip cost

In [ ]:
@tool
def estimate_trip_cost(
    destination: str,
    days: int,
    travelers: int,
    comfort: str = "mid",
) -> Dict[str, Any]:
    """
    Estimate a rough trip budget (SGD) using simple heuristics.
    comfort: budget | mid | premium
    Returns a breakdown and total estimate in SGD.
    """
    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()
    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    # Very rough per-person-per-day estimates (SGD) excluding flights
    lodging_pppd = {"budget": 60, "mid": 140, "premium": 300}[comfort]
    food_pppd = {"budget": 30, "mid": 60, "premium": 120}[comfort]
    local_transport_pppd = {"budget": 10, "mid": 20, "premium": 50}[comfort]
    activities_pppd = {"budget": 20, "mid": 50, "premium": 120}[comfort]

    lodging = lodging_pppd * travelers * days
    food = food_pppd * travelers * days
    transport = local_transport_pppd * travelers * days
    activities = activities_pppd * travelers * days

    subtotal = lodging + food + transport + activities
    contingency = round(subtotal * 0.12)  # 12% buffer
    total = subtotal + contingency

    return {
        "destination": destination,
        "days": days,
        "travelers": travelers,
        "comfort": comfort,
        "currency": "SGD",
        "breakdown": {
            "lodging": lodging,
            "food": food,
            "local_transport": transport,
            "activities": activities,
            "contingency": contingency,
        },
        "total_estimate": total,
        "note": "Heuristic estimate excludes international flights/insurance/visa fees.",
    }

## helper function - Pretty Print

In [ ]:
def pretty_print(response):
    last_msg = response["messages"][-1]

    if isinstance(last_msg.content, list):
        text = "".join(
            block["text"]
            for block in last_msg.content
            if block.get("type") == "text"
        )
    else:
        text = last_msg.content

    print(text)

## 5) Build the LangGraph agent loop

In [ ]:
class State(TypedDict):
    # add_messages makes state["messages"] append-only and compatible with tool loops
    messages: Annotated[List[AnyMessage], add_messages]

tools = [estimate_trip_cost]
web_tool = {"type": "web_search_preview"}  # LangChain docs use web_search_preview
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
llm_with_tools = llm.bind_tools([web_tool] + tools)

def chatbot(state: State):
    # The LLM sees prior messages for this thread via checkpointer
    # We do NOT hardcode tool list in SYSTEM; tool schemas are in bind_tools.
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)  # routes to "tools" when tool calls exist
builder.add_edge("tools", "chatbot")  # tool result goes back to model for next step, this line shall be uncommented for the program to work

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## 4) Run

In [ ]:
# IMPORTANT: use SAME thread_id for multi-turn memory
config1 = {"configurable": {"thread_id": "1"}}

print("\n=== Stage 1: Single Agent (LLM + Memory + Tools) ===\n")

# Turn 1
msg1 = "Plan a 2-day Tokyo trip for 2 adults. Mid comfort. We like food and anime. Avoid very packed schedules."
resp1 = graph.invoke(
    {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": msg1},
    ]},
    config=config1
)
print("--- Turn 1 ---\n")
pretty_print(resp1)

In [ ]:
  # Turn 2 (same thread_id => memory is applied)
msg2 = "Add additional one day trip outside Tokyo and we prefer public transport. and what will be the total cost?"
resp2 = graph.invoke(
    {"messages": [
        {"role": "user", "content": msg2},
    ]},
    config=config1
)

print("\n--- Turn 2 ---\n")
pretty_print(resp2)

In [ ]:
# Turn 3 (memory not applied, new thread)
config2 = {'configurable': {'thread_id': '2'}}
resp3 = graph.invoke(
    {"messages": [
         {"role": "user", "content": msg2}
    ]},
    config=config2
)
print("\n--- Turn 3 ---\n")
pretty_print(resp3)

In [ ]:
# Turn 4 (memory not applied, new thread)
config3 = {'configurable': {'thread_id': '3'}}
resp4 = graph.invoke(
    {"messages": [
         {"role": "system", "content": SYSTEM},
         {"role": "user", "content": msg2},
    ]},
    config=config3
)
print("\n--- Turn 4 ---\n")
pretty_print(resp4)